# AI-Generated Text Detector

This notebook downloads the Kaggle AI vs. Human Text dataset, trains a TF-IDF + Logistic Regression baseline, fine-tunes DistilBERT, and saves evaluation and error-analysis results.

**Before running:** Select **Runtime → Change runtime type → T4 GPU**, then run the code cell below. The default sample is 12,000 rows to fit a free Colab session.

In [ ]:
"""Colab-ready AI-vs-human text classification project.

Recommended Colab setup: Runtime -> Change runtime type -> T4 GPU, then run:
    !python ai_text_detector_colab.py

Optional quick test:
    !MAX_SAMPLES=4000 EPOCHS=1 python ai_text_detector_colab.py
"""

import json
import os
import random
import subprocess
import sys
from pathlib import Path


def install_dependencies():
    packages = [
        "kagglehub>=0.3.4",
        "datasets>=2.20,<4",
        "transformers>=4.44,<5",
        "accelerate>=0.33",
        "scikit-learn>=1.3",
        "pandas>=2.0",
        "matplotlib>=3.7",
        "seaborn>=0.13",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


install_dependencies()

import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)


SEED = int(os.getenv("SEED", "42"))
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "12000"))  # 0 uses all rows
MODEL_NAME = os.getenv("MODEL_NAME", "distilbert-base-uncased")
EPOCHS = int(os.getenv("EPOCHS", "2"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
MAX_LENGTH = int(os.getenv("MAX_LENGTH", "256"))
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "/content/ai_text_detector_results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)


def locate_and_load_dataset():
    dataset_dir = Path(kagglehub.dataset_download("shanegerami/ai-vs-human-text"))
    csv_files = list(dataset_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {dataset_dir}")
    csv_path = max(csv_files, key=lambda p: p.stat().st_size)
    print(f"Loading: {csv_path.name}")
    return pd.read_csv(csv_path)


def standardize_columns(raw):
    lower_to_original = {str(c).lower().strip(): c for c in raw.columns}
    text_candidates = ["text", "essay", "content", "full_text"]
    label_candidates = ["generated", "label", "class", "ai_generated", "is_ai"]
    text_col = next((lower_to_original[x] for x in text_candidates if x in lower_to_original), None)
    label_col = next((lower_to_original[x] for x in label_candidates if x in lower_to_original), None)
    if text_col is None or label_col is None:
        raise ValueError(f"Could not identify text/label columns. Found: {list(raw.columns)}")

    df = raw[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    df = df.dropna().drop_duplicates(subset="text")
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 0]

    if not pd.api.types.is_numeric_dtype(df["label"]):
        normalized = df["label"].astype(str).str.lower().str.strip()
        mapping = {
            "human": 0, "human-written": 0, "0": 0, "false": 0,
            "ai": 1, "ai-generated": 1, "generated": 1, "1": 1, "true": 1,
        }
        df["label"] = normalized.map(mapping)
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].round().astype(int)
    df = df[df["label"].isin([0, 1])].reset_index(drop=True)
    return df


def stratified_sample(df, max_samples):
    if max_samples <= 0 or len(df) <= max_samples:
        return df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    fractions = df["label"].value_counts(normalize=True)
    parts = []
    for label, fraction in fractions.items():
        n = min(int(round(max_samples * fraction)), (df["label"] == label).sum())
        parts.append(df[df["label"] == label].sample(n=n, random_state=SEED))
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)


def save_eda(df):
    df = df.copy()
    df["word_count"] = df["text"].str.split().str.len()
    summary = {
        "rows": int(len(df)),
        "class_counts": {str(k): int(v) for k, v in df["label"].value_counts().items()},
        "word_count_by_class": json.loads(
            df.groupby("label")["word_count"].describe().round(2).to_json()
        ),
    }
    with open(OUTPUT_DIR / "eda_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=df, x="label", ax=axes[0])
    axes[0].set(title="Class distribution", xticklabels=["Human", "AI"])
    clipped = df.assign(word_count=df["word_count"].clip(upper=df["word_count"].quantile(.99)))
    sns.histplot(data=clipped, x="word_count", hue="label", bins=50, element="step", ax=axes[1])
    axes[1].set_title("Text length by class (clipped at 99th percentile)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "eda.png", dpi=160)
    plt.close(fig)


def metric_dict(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def train_baseline(train_df, test_df):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
            min_df=2, max_df=.98, max_features=100000, sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", C=2.0)),
    ])
    model.fit(train_df["text"], train_df["label"])
    predictions = model.predict(test_df["text"])
    joblib.dump(model, OUTPUT_DIR / "tfidf_logistic_regression.joblib")
    return model, predictions, metric_dict(test_df["label"], predictions)


def compute_trainer_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return metric_dict(labels, predictions)


def make_training_arguments():
    common = dict(
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        learning_rate=float(os.getenv("LEARNING_RATE", "2e-5")),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )
    # Transformers renamed evaluation_strategy to eval_strategy.
    try:
        return TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **common)


def train_transformer(train_df, validation_df, test_df):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

    datasets = {}
    for name, frame in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
        ds = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
        datasets[name] = ds.map(tokenize, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label={0: "HUMAN", 1: "AI"},
        label2id={"HUMAN": 0, "AI": 1},
    )
    trainer = Trainer(
        model=model,
        args=make_training_arguments(),
        train_dataset=datasets["train"],
        eval_dataset=datasets["validation"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_trainer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )
    trainer.train()
    output = trainer.predict(datasets["test"])
    predictions = np.argmax(output.predictions, axis=-1)
    save_path = OUTPUT_DIR / "distilbert_detector"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    return trainer, predictions, metric_dict(test_df["label"], predictions)


def save_error_analysis(test_df, predictions, model_name):
    errors = test_df.copy()
    errors["prediction"] = predictions
    errors["error_type"] = np.select(
        [
            (errors["label"] == 0) & (errors["prediction"] == 1),
            (errors["label"] == 1) & (errors["prediction"] == 0),
        ],
        ["false_positive", "false_negative"],
        default="correct",
    )
    errors["word_count"] = errors["text"].str.split().str.len()
    errors[errors["error_type"] != "correct"].sort_values("error_type").to_csv(
        OUTPUT_DIR / f"{model_name}_errors.csv", index=False
    )


def main():
    seed_everything(SEED)
    print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
    if not torch.cuda.is_available():
        print("Warning: enable a GPU in Colab for much faster Transformer training.")

    df = stratified_sample(standardize_columns(locate_and_load_dataset()), MAX_SAMPLES)
    print(f"Usable rows: {len(df):,}\n{df['label'].value_counts().sort_index()}")
    save_eda(df)

    train_df, temp_df = train_test_split(
        df, test_size=.30, random_state=SEED, stratify=df["label"]
    )
    validation_df, test_df = train_test_split(
        temp_df, test_size=.50, random_state=SEED, stratify=temp_df["label"]
    )
    for name, frame in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
        frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

    print("\nTraining TF-IDF + Logistic Regression baseline...")
    _, baseline_pred, baseline_metrics = train_baseline(train_df, test_df)
    save_error_analysis(test_df, baseline_pred, "baseline")

    print("\nFine-tuning Transformer...")
    _, transformer_pred, transformer_metrics = train_transformer(train_df, validation_df, test_df)
    save_error_analysis(test_df, transformer_pred, "transformer")

    results = pd.DataFrame(
        [baseline_metrics, transformer_metrics],
        index=["TF-IDF + Logistic Regression", MODEL_NAME],
    ).round(4)
    results.to_csv(OUTPUT_DIR / "model_comparison.csv")
    print("\nFinal test-set comparison:\n", results)
    print("\nTransformer classification report:\n", classification_report(
        test_df["label"], transformer_pred, target_names=["Human", "AI"], digits=4
    ))

    cm = confusion_matrix(test_df["label"], transformer_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Transformer confusion matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "transformer_confusion_matrix.png", dpi=160)
    plt.close()
    print(f"\nAll models, metrics, plots, splits, and errors saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()
